In [ ]:
import os
import numpy as np
from PIL import Image
from tqdm import tqdm

# ========================================================
# CONFIGURATION
# ========================================================
IMG_SIZE = (300, 300)

# Updated to match the dataset directory in your Kaggle input sidebar
# Adjust this path if your Kaggle dataset mount path differs slightly
REAL_DIR = "/kaggle/input/datasets/anil701/oraldatarwa" 
SYNTH_DIR = "/kaggle/input/datasets/anilk701/synthetic-stress-test-5000-per-class"
CACHE_DIR = "/kaggle/working/cache"
os.makedirs(CACHE_DIR, exist_ok=True)

class_indices = {
    'mdoscc': 0,
    'normal': 1,
    'osmf': 2,
    'pdoscc': 3,
    'wdoscc': 4
}

def cache_dataset(data_dir, prefix):
    images, labels = [], []
    
    for class_name, idx in class_indices.items():
        class_path = os.path.join(data_dir, class_name)
        if not os.path.exists(class_path):
            print(f"⚠️ Warning: Directory missing -> {class_path}")
            continue
        
        # Collect all image files recursively across subfolders using os.walk
        file_paths = []
        for root, _, files in os.walk(class_path):
            for f in files:
                if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                    file_paths.append(os.path.join(root, f))
        
        if len(file_paths) == 0:
            print(f"⚠️ Warning: No images found inside {class_path}")

        for img_path in tqdm(file_paths, desc=f"Caching {prefix.upper()} [{class_name}]"):
            try:
                img = Image.open(img_path).convert('RGB').resize(IMG_SIZE)
                images.append(np.array(img, dtype=np.uint8))
                labels.append(idx)
            except Exception as e:
                print(f"❌ Error loading {img_path}: {e}")
                
    images_arr = np.array(images, dtype=np.uint8)
    labels_arr = np.array(labels, dtype=np.int64)
    
    np.save(os.path.join(CACHE_DIR, f"{prefix}_images.npy"), images_arr)
    np.save(os.path.join(CACHE_DIR, f"{prefix}_labels.npy"), labels_arr)
    print(f"✅ Successfully cached {prefix}: {images_arr.shape} images, {labels_arr.shape} labels.\n")

# Execute Caching
print("=" * 60)
print("📦 STARTING RECURSIVE DATASET CACHING PIPELINE")
print("=" * 60)

cache_dataset(REAL_DIR, "real")
cache_dataset(SYNTH_DIR, "synth")

print("=" * 60)
print(f"🎉 All cache files successfully compiled in: {CACHE_DIR}")

In [ ]:
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118 -U